## Imports

In [8]:
import os
import pandas as pd

current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

file_path = os.path.join(os.path.dirname(current_dir), 'data', 'combined_air_weather_5_cities.csv')

print(f"File path: {file_path}")

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"DataFrame loaded successfully! Shape: {df.shape}")
    print(df.head())
else:
    print(f"File not found at: {file_path}")

DataFrame loaded successfully! Shape: (87720, 18)
               timestamp_utc       city  latitude  longitude  pm2_5  pm10  \
0  2024-08-02 00:00:00+00:00  Islamabad   33.6844    73.0479   30.6  43.7   
1  2024-08-02 01:00:00+00:00  Islamabad   33.6844    73.0479   31.4  44.9   
2  2024-08-02 02:00:00+00:00  Islamabad   33.6844    73.0479   37.7  54.5   
3  2024-08-02 03:00:00+00:00  Islamabad   33.6844    73.0479   40.5  59.4   
4  2024-08-02 04:00:00+00:00  Islamabad   33.6844    73.0479   37.7  56.2   

       co   no2   so2     o3  us_aqi  temperature  humidity  pressure  \
0  1235.0  32.8   0.8    0.0      80         24.8        94     941.6   
1  1134.0  33.3   3.5   22.0      80         26.2        93     940.9   
2   990.0  33.7   7.3   53.0      82         27.2        88     941.2   
3   843.0  31.5  10.4   85.0      84         28.4        82     942.0   
4   693.0  24.2  12.5  120.0      86         29.3        79     942.3   

   wind_speed  wind_direction  precipitation  cl

## Sort the data by time

In [9]:
df = df.sort_values("timestamp_utc").reset_index(drop=True)

## Extract time features

In [12]:

df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])
df["hour"] = df["timestamp_utc"].dt.hour
df["day"] = df["timestamp_utc"].dt.day
df["day_of_week"] = df["timestamp_utc"].dt.dayofweek
df["month"] = df["timestamp_utc"].dt.month

## Encoding Cities

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df["city"] = encoder.fit_transform(df["city"])

## The Target

In [ ]:
y = df["us_aqi"]

## Input Features

In [13]:
x = df.drop(columns=["us_aqi", "timestamp_utc"])
# all other features other than these 2 will be considered input features for the model

## Splitting Data

In [14]:
# dividing into past and known future, through sorting and spliting
split_index = int(len(df) * 0.8)
train = df.iloc[:split_index]
test = df.iloc[split_index:]

## Separate X and y

### Training

In [15]:
X_train = train.drop(columns=["us_aqi", "timestamp_utc"])
y_train = train["us_aqi"]

### Testing

In [16]:
X_test = test.drop(columns=["us_aqi", "timestamp_utc"])
y_test = test["us_aqi"]

### Check

In [17]:
print("Training samples :", len(X_train))
print("Testing samples  :", len(X_test))

print("\nTraining Features")
print(X_train.head())

print("\nTraining Target")
print(y_train.head())

# The goal is to make the data look like the information your model will have when it's deployed. At prediction time, the model will receive features (weather, pollutants, city, time) and must estimate AQI. By separating X and y now and keeping the train/test split chronological, you're mimicking that real-world scenario and getting a trustworthy estimate of how well the model will perform on future data

Training samples : 70176
Testing samples  : 17544

Training Features
         city  latitude  longitude  pm2_5  pm10      co   no2   so2    o3  \
0   Islamabad   33.6844    73.0479   30.6  43.7  1235.0  32.8   0.8   0.0   
1     Karachi   24.8607    67.0011   21.2  78.9   122.0   4.4   3.1  47.0   
2      Lahore   31.5204    74.3587   35.0  52.6   656.0  25.4  11.3  28.0   
3  Rawalpindi   33.5651    73.0169   30.6  43.7  1235.0  32.8   0.8   0.0   
4    Peshawar   34.0151    71.5249   31.9  45.1   413.0  18.2   4.6  40.0   

   temperature  humidity  pressure  wind_speed  wind_direction  precipitation  \
0         24.8        94     941.6         2.3             321            0.2   
1         27.8        90     996.3        25.7             250            0.0   
2         26.0        95     973.4         3.2              90            0.0   
3         25.0        95     943.1         1.9             292            0.2   
4         25.6        89     960.3         2.9              90 